### Data Preparation for Regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load your feature-engineered dataset
df = pd.read_csv('../data/processed/cleaned_data.csv')

# 2. Filter out undisclosed or zero funding entries for continuous forecasting
df = df[df['Total_Funding_USD'] > 0]

# 3. Setup features (X) and continuous target (y) without target leakage
features_regressor = [
    'Total_Funding_Rounds', 'Unique_Investors_Count', 'Fought_Through_Recession',
    'Age_at_Latest_Round', 'Max_Investor_PageRank', 'Sum_Investor_PageRank',
    'country_code_cleaned', 'Industry_Sector_cleaned'
]

X = df[features_regressor].copy()
y = df['Log_Total_Funding']  # Predicting the continuous scale of funding

# One-hot encode categoricals
X = pd.get_dummies(X, columns=['country_code_cleaned', 'Industry_Sector_cleaned'], drop_first=False)

# 4. Train-Test Split (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Regression dataset configured successfully!")
print("Training feature matrix shape:", X_train.shape)

### Train the Regressor (Random Forest Regressor)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Initialize the XGBoost Regressor
regressor = XGBRegressor(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.08,
    random_state=42
)

# 2. Fit the data
print("Training the forecasting engine... Please wait a brief moment.")
regressor.fit(X_train, y_train)

# 3. Predict and evaluate metrics
y_pred = regressor.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\nForecasting Model Training Complete!")
print(f"R2 Score (Genuine Variance Explained): {r2 * 100:.2f}%")
print(f"Mean Absolute Error (in Log Scale): {mae:.4f}")

### Prediction Plot & Serialization

In [ ]:
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Plot Actual vs Predicted values
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.3, color='#5B8DEF')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title("Actual vs. Predicted Funding Log Scale")
plt.xlabel("Actual Log Funding")
plt.ylabel("Predicted Log Funding")
plt.tight_layout()
plt.show()

# 2. Export the forecasting model object
os.makedirs('../models', exist_ok=True)
model_out_path = '../models/funding_model.pkl'

with open(model_out_path, 'wb') as f:
    pickle.dump(regressor, f)

print(f"Success! Forecasting model saved at:\n{model_out_path}")